In [1]:
import torch
import torch.nn as nn

import sys

sys.path.append("..")

from src.paths import Paths
from src.config import TrainingConfig
from src.utils import get_device, initialize_resnet18
from src.data_manager import DataManager
from src.model_trainer import ModelTrainer

In [2]:
training_config = TrainingConfig()
training_config.epochs = 3
device = get_device()
dataset_name = "mnist"

In [3]:
paths = Paths()
data_manager = DataManager(root=paths.DATA_DIR)

train_loader, val_loader, test_loader = data_manager.get_data_loaders(
    dataset_name=dataset_name,
    batch_size=training_config.batch_size,
    download=False,
)

class_names = data_manager.get_class_names(dataset_name)

print(f"Training data size: {len(train_loader) * 64}")
print(f"Validation data size: {len(val_loader) * 64}")
print(f"Test data size: {len(test_loader) * 64}")

Training data size: 54016
Validation data size: 6016
Test data size: 10048


In [4]:
model = initialize_resnet18()

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=training_config.learning_rate,
)

In [5]:
trainer = ModelTrainer(
    model=model,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    class_names=class_names,
)

In [6]:
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=training_config.epochs,
)

Epoch 1/3
Train - Loss: 0.0703 Acc: 0.9789
Val   - Loss: 0.0336 Acc: 0.9900
-------------------------
Epoch 2/3
Train - Loss: 0.0357 Acc: 0.9893
Val   - Loss: 0.0406 Acc: 0.9883
-------------------------
Epoch 3/3
Train - Loss: 0.0295 Acc: 0.9911
Val   - Loss: 0.0233 Acc: 0.9933
-------------------------


In [10]:
results = trainer.evaluate(test_loader)

print(f"OVERALL ACCURACY: {results['accuracy']:.4f}\n\n")
print(results["classification_report"])

OVERALL ACCURACY: 0.9928


              precision    recall  f1-score   support

           0       0.99      1.00      1.00       980
           1       0.99      1.00      0.99      1135
           2       1.00      1.00      1.00      1032
           3       1.00      0.99      0.99      1010
           4       0.99      0.98      0.99       982
           5       0.99      0.99      0.99       892
           6       1.00      0.99      0.99       958
           7       1.00      0.99      0.99      1028
           8       0.99      1.00      0.99       974
           9       0.98      0.99      0.99      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



In [ ]:
paths.MODEL_DIR.mkdir(exist_ok=True)
trainer.save_model(paths.MNIST_MODEL)

/home/mikolajmalolepszy/Desktop/Projects/mgr/models/resnet18_mnist.pth
